# Baseline model — DummyClassifier

**IT2-14** · Binary sentiment classification (0 = negative, 1 = positive).

Goal: establish a simple reference using three `DummyClassifier` strategies. We follow the sequence **build → fit → predict train → predict test → evaluate**. No metrics files or figures are saved in this commit.

## 1. Load the prepared train and test splits

We use the existing stratified splits so every model is evaluated on the same observations. The dummy classifier does not use text content to learn sentiment; the feature input is only a placeholder.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

root = Path.cwd().resolve()
if not (root / "src" / "evaluate.py").exists():
    root = root.parent
if not (root / "src" / "evaluate.py").exists():
    raise FileNotFoundError("Run this notebook from the project root or notebooks/ directory")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.evaluate import evaluate_train_test

SEED = 1234
train = pd.read_csv(root / "data" / "processed" / "train.csv")
test = pd.read_csv(root / "data" / "processed" / "test.csv")

required_columns = {"comment_id", "text", "clean_text", "label"}
assert required_columns.issubset(train.columns)
assert required_columns.issubset(test.columns)
assert len(train) == 800 and len(test) == 200
assert train["label"].value_counts().to_dict() == {0: 400, 1: 400}
assert test["label"].value_counts().to_dict() == {0: 100, 1: 100}

print("Train:", train.shape, "Test:", test.shape)
print("Train labels:", train["label"].value_counts().sort_index().to_dict())
print("Test labels:", test["label"].value_counts().sort_index().to_dict())

## 2. Prepare labels and placeholder features

`DummyClassifier` ignores feature values when making its baseline predictions. We supply a constant feature column, and keep the true labels separate.

In [ ]:
X_train = np.zeros((len(train), 1))
X_test = np.zeros((len(test), 1))
y_train = train["label"].to_numpy()
y_test = test["label"].to_numpy()

## 3. Build, fit, predict, and evaluate

- **most_frequent:** always predicts the most frequent training class (in a tie, scikit-learn chooses one class).
- **stratified:** samples labels according to the training class distribution.
- **uniform:** samples classes with equal probability.

We fix the random seed so the random baselines are reproducible.

In [ ]:
strategies = ("most_frequent", "stratified", "uniform")
results = []

for strategy in strategies:
    # Build
    model = DummyClassifier(strategy=strategy, random_state=SEED)
    # Fit
    model.fit(X_train, y_train)
    # Predict train
    pred_train = model.predict(X_train)
    # Predict test
    pred_test = model.predict(X_test)
    # Evaluate both splits
    metrics = evaluate_train_test(
        name=f"dummy_{strategy}",
        y_train=y_train,
        p_train=pred_train,
        y_test=y_test,
        p_test=pred_test,
    )
    results.append(metrics)

baseline_results = pd.concat(results, ignore_index=True)
display(baseline_results.round(4))

## 4. Inspect the main baseline metrics

In [ ]:
display(
    baseline_results[
        ["model", "split", "accuracy", "precision", "recall", "f1", "f1_macro", "kappa"]
    ].round(4)
)

reference = baseline_results[
    (baseline_results["model"] == "dummy_most_frequent")
    & (baseline_results["split"] == "test")
].iloc[0]
assert np.isclose(reference["accuracy"], 0.5)
assert np.isclose(reference["kappa"], 0.0)
print("most_frequent test accuracy:", reference["accuracy"])
print("most_frequent test kappa:", reference["kappa"])

## 5. Interpretation

The dataset is balanced: half the examples are positive and half are negative in both splits. Therefore, a model that always predicts one class (`most_frequent`) correctly classifies exactly half the examples, giving **accuracy = 0.50**. Its **Cohen’s kappa = 0** because its agreement is no better than expected from the class frequencies.

`stratified` and `uniform` make random predictions, so their exact scores can vary with the seed and sample size, but accuracy is expected to be around 0.50 and kappa around 0. These models do not learn sentiment patterns from the comments. The baseline provides a simple comparison for the later trained models.

**Next (IT2-15):** persist the baseline metrics, plot a confusion matrix, and discuss the chance-level threshold.